In [ ]:
import os
import pandas as pd

# Load dataset
df = pd.read_csv(os.path.join(os.environ["ICU_DATA_DIR"], "44_Feature_Dataset.csv"))

# 1. Basic dataset information
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)

print(f"Shape: {df.shape}")
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

# 2. Column names and data types
print("\n" + "=" * 60)
print("ALL COLUMNS AND DATA TYPES")
print("=" * 60)

for i, (col, dtype) in enumerate(df.dtypes.items(), 1):
    print(f"{i:2}. {col} ({dtype})")

# 3. First and last rows
print("\n" + "=" * 60)
print("FIRST 5 ROWS")
print("=" * 60)
print(df.head().to_string())

print("\n" + "=" * 60)
print("LAST 5 ROWS")
print("=" * 60)
print(df.tail().to_string())

# 4. Identify potential split columns
print("\n" + "=" * 60)
print("POTENTIAL TRAIN/TEST SPLIT COLUMNS")
print("=" * 60)

split_keywords = [
    "split", "train", "test", "set",
    "partition", "subset", "fold"
]

split_cols = [
    col for col in df.columns
    if any(keyword in col.lower() for keyword in split_keywords)
]

if split_cols:
    for col in split_cols:
        print(f"\nColumn: {col}")
        print(df[col].value_counts(dropna=False))
else:
    print("No obvious train/test split column found.")

# 5. Identify possible target columns
print("\n" + "=" * 60)
print("POTENTIAL TARGET COLUMNS")
print("=" * 60)

target_keywords = [
    "icu", "transfer", "target", "label",
    "outcome", "class", "y"
]

target_cols = [
    col for col in df.columns
    if any(
        col.lower() == keyword or
        keyword in col.lower().split("_")
        for keyword in target_keywords
    )
]

for col in target_cols:
    print(f"\nColumn: {col}")
    print(df[col].value_counts(dropna=False).head(20))

# 6. Low-cardinality columns
print("\n" + "=" * 60)
print("COLUMNS WITH <= 5 UNIQUE VALUES")
print("=" * 60)

for col in df.columns:
    if df[col].nunique(dropna=True) <= 5:
        print(f"\n{col}")
        print(df[col].value_counts(dropna=False).to_string())

# 7. Missing values
print("\n" + "=" * 60)
print("MISSING VALUES")
print("=" * 60)

missing = df.isnull().sum()
print(missing[missing > 0].sort_values(ascending=False))

# 8. Duplicate rows
print("\n" + "=" * 60)
print("DUPLICATE ROWS")
print("=" * 60)

print(f"Exact duplicate rows: {df.duplicated().sum()}")

# 9. Dataset summary
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)

print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")
print(f"Potential split columns: {split_cols}")
print(f"Potential target columns: {target_cols}")

In [ ]:
import pandas as pd

df = pd.read_csv("44_Feature_Dataset.csv")

splits = {
    name: set(group["subject_id"])
    for name, group in df.groupby("dataset_split")
}

print("Train/validation overlap:",
      len(splits["train"] & splits["validation"]))

print("Train/test overlap:",
      len(splits["train"] & splits["test"]))

print("Validation/test overlap:",
      len(splits["validation"] & splits["test"]))

print("\nClass distribution by split:")
print(
    pd.crosstab(
        df["dataset_split"],
        df["icu_transfer_within_24h"],
        normalize="index"
    ).round(4)
)

In [ ]:
from pathlib import Path
import json
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score
)

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
OUTPUT_DIR = Path("no_smote_results")
OUTPUT_DIR.mkdir(exist_ok=True)

FILES = {
    "44_features": "44_Feature_Dataset.csv",
    "original_train": "kidney_final_train_dataset.csv",
    "original_test": "kidney_final_test_dataset.csv",
}

# The 44-feature dataset has an explicitly confirmed target.
TARGET_44 = "icu_transfer_within_24h"

# The original kidney dataset previously used this target.
TARGET_ORIGINAL = "icu_transfer"

EXCLUDE_COLUMNS = [
    "dataset_split",
    "row_id",
    "subject_id",
    "hadm_id",
    "index_time"
]

print("Output directory:", OUTPUT_DIR.resolve())

In [ ]:
def get_xy(df, target):
    if target not in df.columns:
        raise ValueError(
            f"Target '{target}' not found. "
            f"Available columns: {df.columns.tolist()}"
        )

    excluded = set(EXCLUDE_COLUMNS + [target])
    X = df.drop(columns=list(excluded & set(df.columns)))
    y = df[target].astype(int)

    if not set(y.unique()).issubset({0, 1}):
        raise ValueError("Target must contain binary 0/1 labels.")

    return X, y


def load_44_features():
    df = pd.read_csv(FILES["44_features"])

    train = df[df["dataset_split"] == "train"].copy()
    val = df[df["dataset_split"] == "validation"].copy()
    test = df[df["dataset_split"] == "test"].copy()

    # Confirm patient-disjoint partitions.
    if "subject_id" in df.columns:
        groups = [
            set(part["subject_id"])
            for part in [train, val, test]
        ]

        assert not (groups[0] & groups[1])
        assert not (groups[0] & groups[2])
        assert not (groups[1] & groups[2])

    return (
        *get_xy(train, TARGET_44),
        *get_xy(val, TARGET_44),
        *get_xy(test, TARGET_44)
    )


def load_original():
    train_df = pd.read_csv(FILES["original_train"])
    test_df = pd.read_csv(FILES["original_test"])

    # Detect the target if its name differs.
    target = TARGET_ORIGINAL

    if target not in train_df.columns:
        candidates = [
            "icu_transfer_within_24h",
            "icu_transfer",
            "target",
            "label"
        ]
        matches = [
            c for c in candidates
            if c in train_df.columns and c in test_df.columns
        ]

        if len(matches) != 1:
            raise ValueError(
                "Cannot uniquely identify the original dataset target. "
                f"Train columns: {train_df.columns.tolist()}"
            )
        target = matches[0]

    # Split original training data into train/validation.
    if "subject_id" in train_df.columns:
        splitter = GroupShuffleSplit(
            n_splits=1,
            test_size=0.20,
            random_state=RANDOM_STATE
        )

        train_idx, val_idx = next(
            splitter.split(
                train_df,
                groups=train_df["subject_id"]
            )
        )

        fit_df = train_df.iloc[train_idx]
        val_df = train_df.iloc[val_idx]

        if "subject_id" in test_df.columns:
            assert not (
                set(train_df["subject_id"])
                & set(test_df["subject_id"])
            ), "Original train/test files share patient IDs."

    else:
        fit_df, val_df = train_test_split(
            train_df,
            test_size=0.20,
            stratify=train_df[target],
            random_state=RANDOM_STATE
        )
        print(
            "WARNING: Original dataset has no subject_id. "
            "Patient-level separation cannot be verified."
        )

    return (
        *get_xy(fit_df, target),
        *get_xy(val_df, target),
        *get_xy(test_df, target)
    )


datasets = {
    "44_features": load_44_features(),
    "original": load_original()
}

for name, data in datasets.items():
    X_train, y_train, X_val, y_val, X_test, y_test = data

    print(f"\n{name}")
    print("Training:", X_train.shape)
    print("Validation:", X_val.shape)
    print("Testing:", X_test.shape)
    print("Training positive rate:", round(y_train.mean(), 4))
    print("Validation positive rate:", round(y_val.mean(), 4))
    print("Testing positive rate:", round(y_test.mean(), 4))

In [ ]:
def build_preprocessor(X):
    numeric_cols = X.select_dtypes(
        include=["number", "bool"]
    ).columns.tolist()

    categorical_cols = [
        c for c in X.columns if c not in numeric_cols
    ]

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    transformers = [
        ("numeric", numeric_pipeline, numeric_cols)
    ]

    if categorical_cols:
        categorical_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="most_frequent")
            ),
            (
                "encoder",
                OneHotEncoder(handle_unknown="ignore")
            )
        ])

        transformers.append(
            ("categorical", categorical_pipeline, categorical_cols)
        )

    return ColumnTransformer(transformers)


def build_model(model_name, X):
    preprocessing = build_preprocessor(X)

    if model_name == "SVM":
        classifier = LinearSVC(
            C=1.0,
            class_weight=None,
            max_iter=5000,
            random_state=RANDOM_STATE,
            dual=False
        )

    elif model_name == "MLP":
        classifier = MLPClassifier(
            hidden_layer_sizes=(64, 32),
            activation="relu",
            solver="adam",
            alpha=0.0001,
            batch_size=512,
            learning_rate_init=0.001,
            max_iter=150,
            early_stopping=False,
            random_state=RANDOM_STATE
        )

    else:
        raise ValueError("Unknown model")

    return Pipeline([
        ("preprocess", preprocessing),
        ("classifier", classifier)
    ])

In [ ]:
def get_scores(model, X, model_name):
    if model_name == "SVM":
        return model.decision_function(X)

    return model.predict_proba(X)[:, 1]


def select_f1_threshold(y_true, scores):
    precision, recall, thresholds = precision_recall_curve(
        y_true, scores
    )

    f1_values = (
        2 * precision[:-1] * recall[:-1]
        / np.maximum(
            precision[:-1] + recall[:-1],
            1e-12
        )
    )

    best_idx = np.argmax(f1_values)

    return float(thresholds[best_idx])

In [ ]:
def evaluate_model(
    model,
    model_name,
    dataset_name,
    X_test,
    y_test,
    threshold
):
    scores = get_scores(model, X_test, model_name)
    predictions = (scores >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        predictions,
        labels=[0, 1]
    ).ravel()

    specificity = tn / (tn + fp) if (tn + fp) else 0

    results = {
        "Dataset": dataset_name,
        "Model": model_name,
        "SMOTE": "None",
        "Threshold": threshold,
        "AUROC": roc_auc_score(y_test, scores),
        "AUPRC": average_precision_score(y_test, scores),
        "Precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_test, predictions, zero_division=0
        ),
        "Specificity": specificity,
        "Accuracy": accuracy_score(y_test, predictions),
        "Balanced Accuracy": balanced_accuracy_score(
            y_test, predictions
        ),
        "F1": f1_score(
            y_test, predictions, zero_division=0
        ),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
        "Predicted Positive": int(predictions.sum()),
        "Test Samples": len(y_test)
    }

    # Save confusion matrix.
    cm = np.array([[tn, fp], [fn, tp]])

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["No ICU transfer", "ICU transfer"],
        yticklabels=["No ICU transfer", "ICU transfer"]
    )

    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.title(f"{dataset_name} — {model_name} (No SMOTE)")
    plt.tight_layout()

    prefix = f"{dataset_name}_{model_name}_no_smote"

    plt.savefig(
        OUTPUT_DIR / f"{prefix}_confusion_matrix.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()

    # Save predictions and decision scores.
    prediction_df = pd.DataFrame({
        "true_label": np.asarray(y_test),
        "score": scores,
        "predicted_label": predictions
    })

    prediction_df.to_csv(
        OUTPUT_DIR / f"{prefix}_predictions.csv",
        index=False
    )

    return results

In [ ]:
all_results = []

for dataset_name, data in datasets.items():

    X_train, y_train, X_val, y_val, X_test, y_test = data

    for model_name in ["SVM", "MLP"]:

        print("\n" + "=" * 65)
        print(f"DATASET: {dataset_name}")
        print(f"MODEL: {model_name}")
        print("SMOTE: NONE")
        print("=" * 65)

        start_time = time.time()

        model = build_model(model_name, X_train)

        print("Training model...")
        model.fit(X_train, y_train)

        print("Selecting validation threshold...")
        val_scores = get_scores(model, X_val, model_name)

        threshold = select_f1_threshold(
            y_val,
            val_scores
        )

        print(f"Selected threshold: {threshold:.6f}")

        print("Evaluating on held-out test set...")

        results = evaluate_model(
            model=model,
            model_name=model_name,
            dataset_name=dataset_name,
            X_test=X_test,
            y_test=y_test,
            threshold=threshold
        )

        elapsed = time.time() - start_time
        results["Training and Evaluation Time (s)"] = elapsed

        all_results.append(results)

        # Save fitted pipeline.
        joblib.dump(
            model,
            OUTPUT_DIR /
            f"{dataset_name}_{model_name}_no_smote.joblib"
        )

        # Save intermediate results after each experiment.
        pd.DataFrame(all_results).to_csv(
            OUTPUT_DIR / "all_results_no_smote.csv",
            index=False
        )

        print("\nRESULTS")
        for key, value in results.items():
            if isinstance(value, (float, np.floating)):
                print(f"{key}: {value:.4f}")
            else:
                print(f"{key}: {value}")

print("\nAll experiments completed.")

In [ ]:
results_df = pd.DataFrame(all_results)

display_columns = [
    "Dataset",
    "Model",
    "SMOTE",
    "Threshold",
    "AUROC",
    "AUPRC",
    "Precision",
    "Recall",
    "Specificity",
    "Accuracy",
    "Balanced Accuracy",
    "F1",
    "TN",
    "FP",
    "FN",
    "TP",
    "Predicted Positive"
]

final_table = results_df[display_columns]

display(final_table.round(4))

# Save Excel workbook.
with pd.ExcelWriter(
    OUTPUT_DIR / "SVM_MLP_No_SMOTE_Results.xlsx",
    engine="openpyxl"
) as writer:

    final_table.to_excel(
        writer,
        sheet_name="All Results",
        index=False
    )

    for dataset_name in final_table["Dataset"].unique():
        subset = final_table[
            final_table["Dataset"] == dataset_name
        ]

        subset.to_excel(
            writer,
            sheet_name=dataset_name[:31],
            index=False
        )

print("Results saved to:", OUTPUT_DIR.resolve())